In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.timeseries import LombScargle

# Accretion
Can we find the break frequency?

# Directions
1. Work with a partner (make friends, do great science!) Write down who your partner is in your notebook!
2. Download the ZTF alert light curve for [this AGN](https://alerce.online/object/ZTF18aaqzhib)
3. Plot the $g$- and $r$-band light curves. Is the variability amplitude the same in each band, or does it have a color?
4. Compute the Lomb Scargle periodogram. We're looking for the break timecale to be somewhere between 0.1 and 100 days... do see anything obvious? ([L-S isn't a great tool](https://arxiv.org/abs/2501.05886v2) here, so it might not be clear)
5. Compute the Structure Function (**I'll include some sample code and info bits below**)
6. Can you find a break timescale with the SF?
7. Given this timescale, estimate the mass of the black hole. There are *many* ways to do this, including scaling relations using the redshift and luminosity, but I would suggest looking at Figure 19 in [this paper](https://ui.adsabs.harvard.edu/abs/2004MNRAS.348..783M/abstract) and just reading an approximate value off
8. Rename your notebook file to YOUR name(s), export to PDF, turn in via [Dropbox upload link](https://www.dropbox.com/request/m0cEy8nBb2LlkF8tzWKl). (**DUE BEFORE THE FINAL EXAM**)

In [ ]:
# NOTE: the Lomb-Scargle might look *slightly* better if you use
#  .autopower(normalization="psd")
# and work in flux space, not magnitude (flux ~ 10^(delta_mag/-2.5))

In [ ]:
# if you want to figure out how far the AGN is....
# redshift from TNS follow-up spectrum: https://www.wis-tns.org/object/2019cvi
z=0.62

# astropy once again FTW!
from astropy.cosmology import Planck18 as P18
DL = P18.luminosity_distance(z).to('cm').value
DL

In [ ]:
# STRUCTURE FUNCTION TIPS
# Let's say you have arrays of N times and N magnitudes

# You need to compute ALL the delta-times and delta-mags (this is an N^2 comparison)
# here's a tricky way to do that:
dmag = mag[:, None] - mag[None, :]

# This should have shape (N,N)
print(dmag.shape)

# --> you can turn it back into a 1-D array easily if you want (you DO):
# dmag.reshape(-1)

In [ ]:
# now you can plot these delta-time and delta-mags
# probably want to use some absolute values

plt.figure()
plt.scatter(dtime, dmag, s=1, alpha=0.5)
# plt.xlim(.05,200)
# plt.xscale('log')
plt.xlabel('Delta Time (days)')
plt.ylabel('Delta Mag')


In [ ]:
# One defintion of the Structure Function is the STD DEV in each delta-time bin
# Based on our data, I might choose few-day bins. 
bins = np.arange(1, 200, 4)

# (or better yet, some kind of LOG binning!)
# bins = np.logspace(np.log10(0.1), np.log10(200), Nbins)

from scipy.stats import binned_statistic
# I like using this to do binned calculations

My_Stat, b_edges, N_samples = binned_statistic(np.abs(dtime), dmag, statistic='std', bins = bins)

In [ ]:
# A basic parameterized version of the Structure Function, for Damped Random Walk process
def SF_fit(dt, tbreak, SF_inf):
    SF = SF_inf * np.sqrt(1 - np.exp(-dt/tbreak))
    return SF

# which you can fit using scipy.optimize (or by-hand) to the SF in delta-time bins from above